# Laboratorio 9 - Redes Neuronales Artificiales

## Pasaporte / Avance

Este notebook resuelve **solo los puntos 1 al 8** de la seccion de actividades del Laboratorio 9.  
Se reutiliza el mismo conjunto de entrenamiento y prueba del Laboratorio 8 para mantener una comparacion valida con los modelos anteriores.


In [ ]:
from pathlib import Path
import json
import runpy
import pandas as pd
from IPython.display import display, Markdown, Image

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 140)
pd.set_option('display.width', 220)

BASE_DIR = Path.cwd()
SCRIPT_PATH = BASE_DIR / 'lab9_rna_avance.py'
OUTPUT_DIR = BASE_DIR / 'salidas_lab9_rna_avance'
FIG_DIR = OUTPUT_DIR / 'graficas'
CONF_DIR = OUTPUT_DIR / 'confusion_matrices'

runpy.run_path(str(SCRIPT_PATH), run_name='__main__')

with open(OUTPUT_DIR / 'lab8_split_reference.json', 'r', encoding='utf-8') as f:
    split_meta = json.load(f)
with open(OUTPUT_DIR / 'rna_summary.json', 'r', encoding='utf-8') as f:
    summary_meta = json.load(f)

base_validation = pd.read_csv(OUTPUT_DIR / 'rna_base_validation.csv')
base_models = pd.read_csv(OUTPUT_DIR / 'rna_base_models_summary.csv')
tuning_validation = pd.read_csv(OUTPUT_DIR / 'rna_tuning_validation.csv')
tuned_models = pd.read_csv(OUTPUT_DIR / 'rna_tuned_models_summary.csv')
best_comparison = pd.read_csv(OUTPUT_DIR / 'rna_best_model_comparison.csv')

cm_best_test = pd.read_csv(CONF_DIR / 'rna_tanh_32_16_test.csv', index_col=0)
cm_best_tuned = pd.read_csv(CONF_DIR / 'rna_tuned_relu_96_48_fast_test.csv', index_col=0)


## Ejercicio 1. Uso de los mismos conjuntos de entrenamiento y prueba

Se reutilizo exactamente el mismo split del Laboratorio 8 para que la comparacion con SVM y con los demas algoritmos anteriores siga siendo valida.


In [ ]:
split_df = pd.DataFrame([
    ['Origen del split', split_meta['split_source']],
    ['Observaciones totales', split_meta['sample_size']],
    ['Entrenamiento', split_meta['train_size']],
    ['Prueba', split_meta['test_size']],
    ['Random state', split_meta['random_state']],
], columns=['Detalle', 'Valor'])

display(split_df)


## Ejercicio 2. Variable respuesta

La variable respuesta usada para clasificacion fue la variable categorica del precio de la casa, con tres niveles:

- `barata`
- `media`
- `cara`

Esta es la misma variable que se venia usando en los laboratorios anteriores de clasificacion.


## Ejercicio 3. Generacion de dos modelos de RNA con topologias y activaciones distintas

Se generaron tres modelos base de redes neuronales para comparar de forma mas solida. Cada uno usa una topologia distinta y una funcion de activacion diferente o una configuracion distinta de regularizacion:

- `rna_relu_64_32`: topologia `(64, 32)` con activacion `relu`
- `rna_tanh_32_16`: topologia `(32, 16)` con activacion `tanh`
- `rna_relu_128_64`: topologia `(128, 64)` con activacion `relu`


In [ ]:
display(base_validation)
display(base_models[['modelo', 'topologia', 'activacion', 'accuracy_test', 'f1_test', 'diagnostico']])


## Ejercicio 4. Prediccion de la variable respuesta

Los modelos se entrenaron con el conjunto de entrenamiento y luego se usaron para predecir sobre entrenamiento y prueba.  
La metrica principal de comparacion fue `F1 macro`, complementada con `Accuracy`, `Precision macro` y `Recall macro`.


In [ ]:
display(base_models[['modelo', 'accuracy_train', 'accuracy_test', 'precision_macro_test', 'recall_macro_test', 'f1_train', 'f1_test', 'f1_gap']])


## Ejercicio 5. Matrices de confusion

Se generaron matrices de confusion para los modelos de RNA. Aqui se muestran las del mejor modelo por desempeno en test y la del mejor modelo tuneado por validacion interna.


In [ ]:
display(Markdown('**Matriz del mejor RNA por test: `rna_tanh_32_16`**'))
display(cm_best_test)
display(Image(filename=str(FIG_DIR / 'rna_best_confusion.png')))

display(Markdown('**Matriz del mejor RNA tuneado por validacion: `rna_tuned_relu_96_48_fast`**'))
display(cm_best_tuned)


## Ejercicio 6. Comparacion de resultados entre modelos de clasificacion RNA

La comparacion se hizo considerando:

- efectividad (`Accuracy`, `F1 macro`)
- tiempo de procesamiento
- equivocaciones mas frecuentes
- clases mejor y peor clasificadas
- importancia de los errores

En general, la clase mas dificil fue `media`, y los errores mas comunes ocurrieron entre `cara` y `media` o entre `media` y `barata`.


In [ ]:
display(base_models[['modelo', 'accuracy_test', 'f1_test', 'tiempo_entrenamiento_seg', 'tiempo_prediccion_seg', 'mejor_clase_test', 'peor_clase_test', 'confusion_mas_frecuente', 'diagnostico']])
display(Image(filename=str(FIG_DIR / 'rna_modelos_base_f1.png')))


## Ejercicio 7. Analisis de sobreajuste

El sobreajuste se analizo comparando `F1 train` contra `F1 test`, junto con el comportamiento de las matrices de confusion.  
Un modelo se considera mas sobreajustado si aprende demasiado bien el entrenamiento pero cae con claridad en prueba.

En los modelos base:

- `rna_tanh_32_16` logro el mejor `F1 test` y un sobreajuste moderado.
- `rna_relu_128_64` tuvo mejor validacion interna, pero mayor brecha train-test.
- `rna_relu_64_32` quedo un poco mas estable, aunque con menor desempeno que `rna_tanh_32_16`.


In [ ]:
display(base_models[['modelo', 'f1_train', 'f1_test', 'f1_gap', 'diagnostico']])
display(Image(filename=str(FIG_DIR / 'rna_overfit_f1.png')))


## Ejercicio 8. Tuneo del modelo elegido y discusion de mejora sin sobreajustar

Se tomo como punto de partida el mejor modelo por validacion interna y se probaron nuevas configuraciones de topologia, regularizacion y tasa de aprendizaje.  
La idea era mejorar el modelo sin empujarlo a un sobreajuste mayor.

Lo interesante aqui es que el mejor modelo tuneado por validacion (`rna_tuned_relu_96_48_fast`) **no mejoro el desempeno en test**. Al contrario, aumento la brecha entre entrenamiento y prueba y termino mas sobreajustado.

Eso sugiere que, para este pasaporte, el modelo mas razonable para recomendar es `rna_tanh_32_16`, porque obtuvo el mejor `F1 macro` en prueba entre los RNA evaluados y mantuvo un ajuste mas sano que el mejor tuneado por validacion.


In [ ]:
display(tuning_validation)
display(tuned_models[['modelo', 'accuracy_test', 'f1_test', 'f1_gap', 'tiempo_entrenamiento_seg', 'diagnostico']])
display(best_comparison)
display(Image(filename=str(FIG_DIR / 'rna_modelos_tuning_f1.png')))


## Conclusiones del pasaporte

- Se reutilizo el mismo split del Lab 8 para mantener comparabilidad.
- Se construyeron varios modelos RNA de clasificacion con topologias y activaciones distintas.
- El mejor RNA por desempeno en test fue `rna_tanh_32_16`.
- El tuneo mejoro la validacion interna en un caso, pero no mejoro el test y aumento el sobreajuste.
- Para el avance, el modelo recomendado es `rna_tanh_32_16` por balance entre desempeno y estabilidad.


## Archivos generados

Las salidas del avance quedaron en `salidas_lab9_rna_avance/` e incluyen:

- `rna_base_models_summary.csv`
- `rna_tuned_models_summary.csv`
- `rna_best_model_comparison.csv`
- `confusion_matrices/`
- `graficas/`
- `README_lab9_rna_avance.md`
